In [1]:
from deep_translator import GoogleTranslator

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import json

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import re

# Translating Lyrics 

In [ ]:
# translating lyrics of songs 

def translate_lyrics(artist):
    # getting lyrics data 
    file = open(f'raw_data/lyrics/{artist}_lyrics.txt', 'r', errors='ignore')

    song = None
    songs = []
    song_index = -1

    # reading lyrics 
    for line in file: 
        line = line.strip()
        if len(line) > 1: 
            if line.startswith("SONG NAME ["):
                song_index += 1
                songs.append({})
                song = line[len("SONG NAME ["):-1]
                songs[song_index]['title'] = song
                songs[song_index]['lyrics_trans'] = ''

                # goes to previous song and apply translation 
                if song_index > 0:
                    # check if lyrics are too long for the translator (omitted if exceeds max length)
                    if len(songs[song_index-1]['lyrics_trans']) > 5000:
                        songs[song_index-1]['lyrics_trans'] = None
                    else:
                        songs[song_index-1]['lyrics_trans'] = GoogleTranslator(source='auto', target='en').translate(text=songs[song_index-1]['lyrics_trans'])

                print(f'({song_index}): {song}')

            elif not 'Contributors' in line and not 'Read More' in line:
                songs[song_index]['lyrics_trans'] += (line + ' ')

    # for the last index 
    if len(songs[song_index]['lyrics_trans']) > 5000:
        songs[song_index]['lyrics_trans'] = None
    else:
        songs[song_index]['lyrics_trans'] = GoogleTranslator(source='auto', target='en').translate(text=songs[song_index]['lyrics_trans'])

    fileDump = open(f'{artist}_lyrics_trans.json','w')
    json.dump(songs, fileDump, indent=4)

In [ ]:
# files then moved to: translated_lyrics/
artists = ['blackpink','exo','twice']
for artist in artists:
    translate_lyrics(artist)

(0): 마지막처럼 (AS IF IT’S YOUR LAST)
(1): 마지막처럼 (AS IF IT’S YOUR LAST) (Inkigayo Remix)
(2): AS IF IT’S YOUR LAST (Japanese Version)
(3): As If It’s Your Last (Japan Version / BLACKPINK 2019-2020 WORLD TOUR IN YOUR AREA - TOKYO DOME)
(4): AS IF IT’S YOUR LAST -JP Ver.- (BLACKPINK ARENA TOUR 2018 ”SPECIAL FINAL IN KYOCERA DOME OSAKA”)
(5): As If It’s Your Last (Live)
(6): As If It’s Your Last (THE SHOW Live)
(7): Awesome Screen, Awesome Camera
(8): Bet You Wanna
(9): 붐바야 (BOOMBAYAH)
(10): BOOMBAYAH (Japan Version / BLACKPINK 2019-2020 WORLD TOUR IN YOUR AREA - TOKYO DOME)
(11): BOOMBAYAH -JP Ver.-
(12): BOOMBAYAH -JP Ver.- (BLACKPINK ARENA TOUR 2018 ”SPECIAL FINAL IN KYOCERA DOME OSAKA”)
(13): BOOMBAYAH (Live)
(14): BOOMBAYAH (THE SHOW Live)
(15): Crazy Over You
(16): Crazy Over You (Live)
(17): 뚜두뚜두 (DDU-DU DDU-DU)
(18): DDU-DU DDU-DU (Japan Version / BLACKPINK 2019-2020 WORLD TOUR IN YOUR AREA - TOKYO DOME)
(19): DDU-DU DDU-DU (JP Ver.)
(20): DDU-DU DDU-DU -JP Ver.- (BLACKPINK ARENA TOUR

In [1]:
# RequestError --> saved whatever it loaded

# artist = 'bts'
# file = open(f'raw_data/lyrics/{artist}_lyrics.txt', 'r', errors='ignore')

# song = None
# songs = []
# song_index = -1

# for line in file: 
#     line = line.strip()
#     if len(line) > 1: 
#         if line.startswith("SONG NAME ["):
#             song_index += 1
#             songs.append({})
#             song = line[len("SONG NAME ["):-1]
#             songs[song_index]['title'] = song
#             songs[song_index]['lyrics_trans'] = ''

#             if song_index > 0:
#                 if len(songs[song_index-1]['lyrics_trans']) > 5000:
#                     songs[song_index-1]['lyrics_trans'] = None
#                 else:
#                     songs[song_index-1]['lyrics_trans'] = GoogleTranslator(source='auto', target='en').translate(text=songs[song_index-1]['lyrics_trans'])

#             print(f'({song_index}): {song}')

#         elif not 'Contributors' in line and not 'Read More' in line:
#             songs[song_index]['lyrics_trans'] += (line + ' ')

# # for the last index 
# if len(songs[song_index]['lyrics_trans']) > 5000:
#     songs[song_index]['lyrics_trans'] = None
# else:
#     songs[song_index]['lyrics_trans'] = GoogleTranslator(source='auto', target='en').translate(text=songs[song_index]['lyrics_trans'])

# fileDump = open(f'{artist}_lyrics_trans.json','w')
# json.dump(songs, fileDump, indent=4)

In [ ]:
# saving partially loaded translations from prev cell 

# fileDump = open(f'{artist}_lyrics_trans.json','w')
# json.dump(songs, fileDump, indent=4)

# Scoring Lyrics 

In [ ]:
def score_lyrics(artist):
    # getting data 
    file = open(f'translated_lyrics/{artist}_lyrics_trans.json', 'r')
    data = json.load(file)

    # getting stopwords (frequent neutral words)
    stopwords = nltk.corpus.stopwords.words("english")

    # creating sentiment intensity analyzer
    sia = SentimentIntensityAnalyzer()

    # iterating through the songs
    for index, datapoint in enumerate(data):
        lyrics_filtered = []

        # expanding the lyrics (string) into a list to filter stopwords 
        if datapoint['lyrics_trans'] != None and len(datapoint['lyrics_trans']) > 0:
            lyrics = datapoint['lyrics_trans'].split(' ')
            # filtering out stopwords and symbols 
            for word in lyrics:
                word = re.sub(r'[()\[\],:.""]','',word).lower()
                if word not in stopwords:
                    lyrics_filtered.append(word)

        # joining the filtered lyrics back into a string
        lyrics_filtered = ' '.join(lyrics_filtered)
        data[index]['lyrics_filtered'] = lyrics_filtered

        # scoring the lyrics 
        score = sia.polarity_scores(lyrics_filtered)
        data[index]['sentiment'] = score

    fileDump = open(f'{artist}_lyrics_scored.json','w')
    json.dump(data, fileDump, indent=4)


In [ ]:
# files then moved to: scored/
artists = ['bts','blackpink','exo','twice']
for artist in artists:
    score_lyrics(artist)